# Financial Fraud Detection with Blind Insight Data

This notebook demonstrates how to use financial fraud data from Blind Insight for machine learning tasks with scikit-learn.

## Prerequisites

1. Install required packages:
   ```bash
   pip install -r requirements.txt
   ```

2. Ensure the backend API is running:
   ```bash
   cd cube-server
   node index.js
   ```

3. Make sure you have uploaded the financial fraud dataset to Blind Insight using the importer tool.


## Setup and Imports


In [21]:
import matplotlib.pyplot as plt
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# Import the Blind Insight client
import sys
sys.path.append('.')
from blind_insight_client import BlindInsightClient, load_fraud_from_blind


## Load Data from Blind Insight

Instead of using standard datasets, we load financial fraud data from Blind Insight.

**Note**: Update the organization, dataset_slug, and schema_slug to match your Blind Insight setup.


### Workflows
- **Encrypted-only (recommended)**: Use Blind Insight encrypted primitives (count, avg, min, max, ranges) and keep rows encrypted. See the *Encrypted Aggregation Classifier* section below.
- **Plaintext ML (illustrative only)**: Decrypt data and run scikit-learn directly. This is not encrypted and is kept for reference/testing.


In [22]:
# Configuration - Plaintext ML example (illustrative)
# NOTE: This decrypts data; for encrypted-only workflows see the section below.
ORGANIZATION = "demo"
DATASET_SLUG = "FraudAnalysis"
SCHEMA_SLUG = "FraudAnalysis"
API_URL = "https://proxy.local.blindinsight.io/"

# Authentication for backend API (required for metadata lookups like schema ID resolution)
# If you get 403 errors, provide your Blind Insight credentials here:
USERNAME = "data_owner@localhost"  # Your Blind Insight account email
PASSWORD = "blindinsight"  # Your Blind Insight account password

# Schema ID (for encrypted datasets, provide this directly to avoid metadata API calls)
SCHEMA_ID = "oKVxDa56fi4EwrgjpNiibw"  # Schema ID for the encrypted fraud dataset

# Plaintext load (decrypt=True inside load_fraud_from_blind)
# Load fraud detection data from Blind Insight
# IMPORTANT: If your data is encrypted, decrypt=True is REQUIRED to get plaintext field names.
# Without decryption, field names are hashed and won't match column names.
# Available columns: amount, country, device_risk_score, hour, ip_risk_score, 
# is_fraud, merchant_category, transaction_id, transaction_type, user_id
X, y = load_fraud_from_blind(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    api_url=API_URL,
    decrypt=True,  # REQUIRED for encrypted data - decrypts to get plaintext field names
    feature_cols=["amount", "device_risk_score", "ip_risk_score", "hour"],  # Transaction fraud features
    target_col="is_fraud",  # Binary fraud indicator (0 = not fraud, 1 = fraud)
    username=USERNAME,  # Authentication for schema lookup
    password=PASSWORD,  # Authentication for schema lookup
    verify_ssl=False,  # Disable SSL verification for local development with self-signed certs
    schema_id=SCHEMA_ID  # Use schema ID directly to avoid metadata API lookup
)

print(f"Loaded {X.shape[0]} samples with {X.shape[1]} features")
print(f"Feature shape: {X.shape}")
print(f"Target shape: {y.shape}")
print(f"Unique classes: {np.unique(y)}")
print(f"Class distribution: {np.bincount(y)}")
print(f"\nNote: Binary classification - 0 = not fraud, 1 = fraud")
print(f"Features used: amount, device_risk_score, ip_risk_score, hour")


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Expanded DataFrame columns after flattening 'data': ['amount', 'country', 'device_risk_score', 'hour', 'ip_risk_score', 'is_fraud', 'merchant_category', 'transaction_id', 'transaction_type', 'user_id']
Available columns in DataFrame: ['amount', 'country', 'device_risk_score', 'hour', 'ip_risk_score', 'is_fraud', 'merchant_category', 'transaction_id', 'transaction_type', 'user_id']
Using feature columns: ['amount', 'device_risk_score', 'ip_risk_score', 'hour']
Using target column: is_fraud
Loaded 9880 samples with 4 features
Feature shape: (9880, 4)
Target shape: (9880,)
Unique classes: [0 1]
Class distribution: [9386  494]

Note: Binary classification - 0 = not fraud, 1 = fraud
Features used: amount, device_risk_score, ip_risk_score, hour


## Prepare Data for Classification

For fraud detection, we'll use transaction features: amount, device_risk_score, and ip_risk_score. 
The target variable 'is_fraud' is already binary (0 = not fraud, 1 = fraud), so we can use all data directly.


In [23]:
# For fraud detection, use transaction features
# The target 'is_fraud' is already binary (0 = not fraud, 1 = fraud)
# Use all features: amount, device_risk_score, ip_risk_score, hour
X_subset = X  # Use all features
y_subset = y  # Already binary (0/1)

# Optionally, select a subset of features for simpler model
# Using: amount (index 0), device_risk_score (index 1), ip_risk_score (index 2)
X_subset = X_subset[:, [0, 1, 2]]  # amount, device_risk_score, ip_risk_score

print(f"Data shape: {X_subset.shape}")
print(f"Target classes: {np.unique(y_subset)}")
print(f"Class distribution: {np.bincount(y_subset)}")
print(f"Features used: amount, device_risk_score, ip_risk_score")


Data shape: (9880, 3)
Target classes: [0 1]
Class distribution: [9386  494]
Features used: amount, device_risk_score, ip_risk_score


In [24]:
print(f"Sample features (amount, device_risk_score, ip_risk_score):\n{X_subset[:10]}")
print(f"\nSample targets:\n{y_subset[:10]}")
print(f"\nFeature statistics:")
print(f"Amount - Min: {X_subset[:, 0].min()}, Max: {X_subset[:, 0].max()}, Mean: {X_subset[:, 0].mean():.2f}")
print(f"Device risk score - Min: {X_subset[:, 1].min()}, Max: {X_subset[:, 1].max()}, Mean: {X_subset[:, 1].mean():.2f}")
print(f"IP risk score - Min: {X_subset[:, 2].min()}, Max: {X_subset[:, 2].max()}, Mean: {X_subset[:, 2].mean():.2f}")


Sample features (amount, device_risk_score, ip_risk_score):
[[6.39928700e+01 4.37436215e-02 2.22846612e-01]
 [1.13305751e+02 1.42162983e-01 3.48855433e-02]
 [1.31963497e+02 3.98973275e-02 4.45114424e-02]
 [4.28417797e+03 7.37854234e-01 7.31070697e-01]
 [7.66954798e+01 9.39759661e-02 1.84998839e-01]
 [1.38641702e+02 1.08045629e-01 2.09671007e-01]
 [1.33263575e+02 5.74340877e-02 9.21888215e-02]
 [1.75512081e+01 1.24694196e-02 1.75071583e-01]
 [1.00000000e+00 1.06583552e-01 1.80465035e-01]
 [1.54718043e+02 1.64080897e-01 3.09203772e-02]]

Sample targets:
[0 0 0 1 0 0 0 0 0 0]

Feature statistics:
Amount - Min: 1.0, Max: 11628.213880743357, Mean: 177.86
Device risk score - Min: 3.0274785175044003e-05, Max: 0.998736630233235, Mean: 0.18
IP risk score - Min: 8.804990376776178e-06, Max: 0.9996026492679309, Mean: 0.18


## Train Logistic Regression Model


In [25]:
# Create an instance of Logistic Regression Classifier and fit the data
logreg = LogisticRegression(C=1e5, max_iter=1000)
clf = logreg.fit(X_subset, y_subset)

print("Model trained successfully!")
print(clf)


Model trained successfully!
LogisticRegression(C=100000.0, max_iter=1000)


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/linear_model/_linear_loss.py:203: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept


## Evaluate Model Accuracy


In [26]:
# Make predictions
y_pred = clf.predict(X_subset)

# Calculate accuracy
accuracy = accuracy_score(y_subset, y_pred)
print(f"Accuracy: {accuracy:.4f} ({accuracy * 100:.2f}%)")


Accuracy: 1.0000 (100.00%)


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: divide by zero encountered in matmul
  ret = a @ b
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: overflow encountered in matmul
  ret = a @ b
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/sklearn/utils/extmath.py:227: RuntimeWarning: invalid value encountered in matmul
  ret = a @ b


Blind 

In [27]:
# Example: Use encrypted filters to narrow dataset before decryption
# Encrypted aggregation-based classifier (no decryption)
# Binary fraud detection using amount feature
# 
# NOTE: The proxy doesn't support count aggregations on float fields like "amount".
# Workaround: Use range filters with query() to count records, or use avg/min/max aggregations.
# Alternative: Use integer fields or integer-scaled values for count aggregations.

from blind_insight_client import BlindInsightClient
import requests
from urllib.parse import urlencode

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
feature = "amount"  # Use transaction amount as the feature
class_a = "0"  # Not fraud (is_fraud = 0)
class_b = "1"  # Fraud (is_fraud = 1)

# Helper to extract aggregation value (supports both shapes)
# Shape A: {records: [{data: {value: X}}]}
# Shape B: {records: [{value: X, aggregation_type: ...}]}

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    # Shape A
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    # Shape B
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# WORKAROUND: Since count() doesn't work on float fields via proxy, we use a different approach:
# 1. Use query() with range filters to get record counts
# 2. Or use avg/min/max aggregations instead of count
# 3. Or use the backend API directly (requires hashing, more complex)

# Method 1: Use query() with range filters to count records
# This works because query() supports range filters on floats, we just count the results
def count_records_with_filter(extra_filters, amount_range=None):
    """Count records by querying with filters and counting results."""
    filters = extra_filters.copy() if extra_filters else []
    if amount_range:
        filters.append(f"{feature}:{amount_range}")
    
    result = client.query(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        limit=10000,  # Large limit to get all matching records
        offset=0,
        filters=filters,
        decrypt=False,
        schema_id=SCHEMA_ID
    )
    records = result.get("records", [])
    return len(records)

# Precompute totals per class using query() instead of count aggregation
print("Counting records per class using query() method (workaround for float count limitation)...")
count_a_total = count_records_with_filter([f"is_fraud:{class_a}"])
count_b_total = count_records_with_filter([f"is_fraud:{class_b}"])

print(f"Total class A (not fraud): {count_a_total}")
print(f"Total class B (fraud): {count_b_total}")

# Thresholds over transaction amount
# Adjust range based on your data - example: 0 to 10000 with step 500
thresholds = range(0, 10001, 500)  # Adjust based on your amount range

best_acc = -1
best_t = None
best_counts = None

print(f"\nSearching {len(thresholds)} thresholds...")
for t in thresholds:
    # Count records with amount < threshold for each class
    count_a_left = count_records_with_filter(
        [f"is_fraud:{class_a}"],
        amount_range=f"<{t}"
    )
    count_b_left = count_records_with_filter(
        [f"is_fraud:{class_b}"],
        amount_range=f"<{t}"
    )

    # Accuracy: predict class_a if amount < threshold, else class_b
    correct = count_a_left + (count_b_total - count_b_left)
    total = count_a_total + count_b_total
    acc = correct / total if total else 0.0

    if acc > best_acc:
        best_acc = acc
        best_t = t
        best_counts = (count_a_left, count_b_left, correct, total)

print(f"\nBest threshold on encrypted counts: {best_t:.3f}")
print(f"Encrypted-rule accuracy: {best_acc:.4f} ({best_acc*100:.2f}%)")
print(f"Counts at best threshold -> class_a_left: {best_counts[0]:.0f}, class_b_left: {best_counts[1]:.0f}, correct: {best_counts[2]:.0f}/{best_counts[3]:.0f}")

print("\nRule: predict not fraud (0) if amount < threshold, else fraud (1) (encrypted counts only)")
print("\nNote: This uses query() with range filters instead of count() aggregation due to proxy limitation on float fields.")


Counting records per class using query() method (workaround for float count limitation)...


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Total class A (not fraud): 10000
Total class B (fraud): 10000

Searching 21 thresholds...


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedoc


Best threshold on encrypted counts: 0.000
Encrypted-rule accuracy: 0.5000 (50.00%)
Counts at best threshold -> class_a_left: 10000, class_b_left: 10000, correct: 10000/20000

Rule: predict not fraud (0) if amount < threshold, else fraud (1) (encrypted counts only)

Note: This uses query() with range filters instead of count() aggregation due to proxy limitation on float fields.


## Alternative: Using the Client Directly

You can also use the client directly for more control over the data loading process.


## Encrypted Aggregation Classifier (no decryption)

This section demonstrates classification using only Blind Insight's encrypted primitives:
- Use encrypted filters and aggregations (count/min/max/ranges) on encrypted data
- Never decrypt rows
- Build a simple rule-based classifier using aggregate stats only

**Dataset note**: For this demo we assume the numeric features are integer-scaled (e.g., original floats multiplied by 10) so that counts/ranges operate on integers. Adjust ranges accordingly if you change the scaling.

Approach (two-class demo: setosa vs versicolor):
1. Search thresholds over integer-scaled petal-length using encrypted `count(<t)` per class.
2. Pick the threshold with best accuracy on encrypted counts.
3. No row-level decryption; only aggregates are returned.


## Encrypted averages: transaction amount per fraud class
Use Blind Insight encrypted `avg` to compute the mean transaction amount for fraud vs non-fraud transactions.

**Note**: The proxy doesn't support `avg()` aggregations on float fields. If you get a 422 error, this is expected. See cell 15 for a workaround using `query()` with range filters.


In [28]:
# Encrypted avg of amount per fraud class
# Compute the mean transaction amount for fraud vs non-fraud transactions
from blind_insight_client import BlindInsightClient

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
feature = "amount"  # Transaction amount field
classes = ["0", "1"]  # is_fraud: 0 = not fraud, 1 = fraud
class_labels = {"0": "Not Fraud", "1": "Fraud"}

# Helper to extract aggregation value (supports both shapes)
def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# NOTE: avg() aggregation on float fields may not work via proxy. 
# If you get an error, this is expected - the proxy doesn't support aggregations on float fields.
# You can use the workaround from cell 15 (using query() with range filters) instead.

for cls in classes:
    try:
        resp = client.aggregate(
            organization=ORGANIZATION,
            dataset_slug=DATASET_SLUG,
            schema_slug=SCHEMA_SLUG,
            agg_filter=f"{feature}:avg(0~100000)",  # Range for amount field
            extra_filters=[f"is_fraud:{cls}"],
            decrypt=False,  # keep encrypted; avg is computed inside Blind Insight
            schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
        )
        mean_val = agg_value(resp)
        print(f"Mean {feature} for {class_labels[cls]} (is_fraud={cls}): {mean_val:.2f}")
    except ValueError as e:
        if "aggregation operations are not supported for float values" in str(e) or "422" in str(e):
            print(f"Note: avg() aggregation on float field '{feature}' is not supported via proxy.")
            print(f"  This is a known limitation. Use query() with range filters (see cell 15) as a workaround.")
            print(f"  Error: {str(e)[:200]}")
        else:
            raise



Note: avg() aggregation on float field 'amount' is not supported via proxy.
  This is a known limitation. Use query() with range filters (see cell 15) as a workaround.
  Error: Aggregation query failed (422 Unprocessable Entity). This might mean:
1. The aggregation expression format is not supported: amount:avg(0~100000)
2. The proxy might not be configured to handle aggrega


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


Note: avg() aggregation on float field 'amount' is not supported via proxy.
  This is a known limitation. Use query() with range filters (see cell 15) as a workaround.
  Error: Aggregation query failed (422 Unprocessable Entity). This might mean:
1. The aggregation expression format is not supported: amount:avg(0~100000)
2. The proxy might not be configured to handle aggrega


## Encrypted averages per batch (example)
Compute means for numeric features in batches using encrypted `avg`. 

**Note**: This example uses `dataset-order` which may not exist in all datasets. Also, `avg()` aggregations on float fields are not supported via proxy. This cell is provided as an example that may need adaptation for your specific dataset.


In [29]:
# Batch means over all rows, in batches
# NOTE: This cell requires a "dataset-order" field which may not exist in your dataset.
# This is an advanced example that demonstrates batch processing. You may need to skip this cell
# if your dataset doesn't have a dataset-order field, or adapt it to use a different ordering field.

# Make sure you've run the data loading cell (cell 5) first to define ORGANIZATION, DATASET_SLUG, 
# SCHEMA_SLUG, API_URL, USERNAME, PASSWORD, and SCHEMA_ID variables
from blind_insight_client import BlindInsightClient

# Verify required variables are defined (check in globals() to avoid NameError)
required_vars = ['ORGANIZATION', 'DATASET_SLUG', 'SCHEMA_SLUG', 'API_URL', 'USERNAME', 'PASSWORD', 'SCHEMA_ID']
missing_vars = [var for var in required_vars if var not in globals() or not globals()[var]]
if missing_vars:
    raise ValueError(
        f"The following variables are not defined: {', '.join(missing_vars)}. "
        f"Please run the data loading cell (cell 5) first to define these variables."
    )

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)
# Use fraud dataset numeric fields - adjust ranges based on your data
features = ["amount", "device_risk_score", "ip_risk_score", "hour"]  # Fraud dataset numeric fields
batch_size = 10

print("Note: This cell requires 'dataset-order' field and avg() aggregations on floats.")
print("If you get errors, this is expected - the proxy doesn't support these operations.")
print("You can skip this cell or adapt it for your specific dataset.\n")

# Reuse agg_value if defined; otherwise define here

def agg_value(resp):
    recs = resp.get("records", [])
    if not recs:
        raise ValueError(f"Unexpected aggregation response: {resp}")
    rec0 = recs[0]
    if "data" in rec0 and isinstance(rec0["data"], dict) and "value" in rec0["data"]:
        v = rec0["data"].get("value")
        return float(v) if v is not None else 0.0
    if "value" in rec0:
        v = rec0.get("value")
        return float(v) if v is not None else 0.0
    raise ValueError(f"Unexpected aggregation response shape: {resp}")

# This cell demonstrates batch processing using dataset-order field
# NOTE: The fraud dataset may not have a "dataset-order" field, so this cell may not work.
# This is an advanced example that requires a dataset with an ordering field.

try:
    # Determine total rows via encrypted count on dataset-order
    # NOTE: This will fail if dataset-order field doesn't exist in your schema
    TOTAL_MAX = 100000  # a large upper bound
    count_resp = client.aggregate(
        organization=ORGANIZATION,
        dataset_slug=DATASET_SLUG,
        schema_slug=SCHEMA_SLUG,
        agg_filter=f"dataset-order:count(0~{TOTAL_MAX})",
        decrypt=False,
        schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
    )
    total_rows = int(agg_value(count_resp))
    print(f"Total rows detected: {total_rows}")

    start = 0
    while start < total_rows:
        end = min(start + batch_size - 1, total_rows - 1)
        batch_filter = f"dataset-order:{start}~{end}"
        print(f"\nBatch {start}-{end}:")
        for feature in features:
            try:
                resp = client.aggregate(
                    organization=ORGANIZATION,
                    dataset_slug=DATASET_SLUG,
                    schema_slug=SCHEMA_SLUG,
                    agg_filter=f"{feature}:avg(0~100000)",  # Adjust range for fraud dataset
                    extra_filters=[batch_filter],
                    decrypt=False,
                    schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
                )
                mean_val = agg_value(resp)
                print(f"  mean {feature}: {mean_val:.3f}")
            except ValueError as e:
                if "422" in str(e) or "aggregation operations are not supported" in str(e):
                    print(f"  Note: avg() on '{feature}' not supported (float field limitation)")
                else:
                    raise
        start += batch_size
except ValueError as e:
    if "label not found" in str(e) or "dataset-order" in str(e):
        print("=" * 60)
        print("SKIPPED: This cell requires a 'dataset-order' field which doesn't exist")
        print("in the fraud dataset. This is an advanced example for datasets that")
        print("have an ordering field.")
        print("=" * 60)
        print(f"\nError details: {str(e)[:200]}")
    else:
        raise



Note: This cell requires 'dataset-order' field and avg() aggregations on floats.
If you get errors, this is expected - the proxy doesn't support these operations.
You can skip this cell or adapt it for your specific dataset.

SKIPPED: This cell requires a 'dataset-order' field which doesn't exist
in the fraud dataset. This is an advanced example for datasets that
have an ordering field.

Error details: Aggregation query failed (422 Unprocessable Entity). This might mean:
1. The aggregation expression format is not supported: dataset-order:count(0~100000)
2. The proxy might not be configured to handl


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [30]:
# Example: Load data as a pandas DataFrame for exploration
# Make sure you've run the data loading cell (cell 5) first to define required variables
from blind_insight_client import BlindInsightClient

# Verify required variables are defined
required_vars = ['ORGANIZATION', 'DATASET_SLUG', 'SCHEMA_SLUG', 'API_URL', 'USERNAME', 'PASSWORD', 'SCHEMA_ID']
missing_vars = [var for var in required_vars if var not in globals() or not globals()[var]]
if missing_vars:
    raise ValueError(
        f"The following variables are not defined: {', '.join(missing_vars)}. "
        f"Please run the data loading cell (cell 5) first to define these variables."
    )

client = BlindInsightClient(api_url=API_URL, username=USERNAME, password=PASSWORD, verify_ssl=False)

# Check API health
health = client.health_check()
print(f"API Status: {health}")

# Load full dataset as DataFrame
# Pass schema_id to avoid metadata lookup that requires authentication
df = client.load_data(
    organization=ORGANIZATION,
    dataset_slug=DATASET_SLUG,
    schema_slug=SCHEMA_SLUG,
    limit=150,
    schema_id=SCHEMA_ID  # Use schema ID directly to avoid lookup
)

print(f"\nDataFrame shape: {df.shape}")
print(f"\nColumn names: {df.columns.tolist()}")
print(f"\nFirst few rows:")
print(df.head())
print(f"\nDataFrame info:")
print(df.info())


API Status: {'status': 'ok', 'api_version': '10.8.6', 'behind_proxy': True}


/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(
/Users/ivan/code/blindinsight/server/.venv/lib/python3.11/site-packages/urllib3/connectionpool.py:1097: InsecureRequestWarning: Unverified HTTPS request is being made to host 'proxy.local.blindinsight.io'. Adding certificate verification is strongly advised. See: https://urllib3.readthedoc


DataFrame shape: (150, 4)

Column names: ['data', 'id', 'schema', 'url']

First few rows:
                                                data                      id  \
0  {'amount': 63.99286999198463, 'country': 'FR',...  bVAmQ4kTHV68Eqk7beEF2o   
1  {'amount': 113.30575124310727, 'country': 'TR'...  LfJn5wf8jii5XAH9YHvSzL   
2  {'amount': 131.96349737388977, 'country': 'FR'...  3M3HdVPtxSNFCfg4KvV7HJ   
3  {'amount': 4284.177966789535, 'country': 'TR',...  F9HgpSLQbqG7rnyWtehtif   
4  {'amount': 76.69547978824451, 'country': 'TR',...  TKaJVc7yCaVM8ghjrLP6JN   

                                              schema  \
0  https://localhost:8080/api/schemas/oKVxDa56fi4...   
1  https://localhost:8080/api/schemas/oKVxDa56fi4...   
2  https://localhost:8080/api/schemas/oKVxDa56fi4...   
3  https://localhost:8080/api/schemas/oKVxDa56fi4...   
4  https://localhost:8080/api/schemas/oKVxDa56fi4...   

                                                 url  
0  https://localhost:8080/api/record

## Summary

This notebook demonstrates:

1. ✅ Loading data from Blind Insight instead of sklearn datasets
2. ✅ Using the data with scikit-learn for machine learning
3. ✅ Training a logistic regression classifier
4. ✅ Evaluating model accuracy

The key difference from the standard scikit-learn example is that instead of:
```python
iris = datasets.load_iris()
```

We use:
```python
X, y = load_iris_from_blind(organization, dataset_slug, schema_slug)
```

This allows data scientists to work with encrypted, privacy-preserving data stored in Blind Insight while using standard ML libraries and workflows.
